In [1]:
import numpy as np
import pandas as pd
import cvxpy as cp
import seaborn as sns
import mosek
import matplotlib.pyplot as plt
import datetime as date
from datetime import datetime as dt
from dateutil.relativedelta import *
import scipy.stats
from scipy.stats import rankdata
import distortion_function as hf
import phi_divergence as phi
import affine_approx as af
import Utility_functions as ut
import Cutting_plane as ct
from time import process_time
import Hit_and_Run as hr

In [2]:
def make_RC_Var(N,M,I):
    v = cp.Variable(M)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N)
    z = cp.Variable(M)
    s = cp.Variable(N)
    w = cp.Variable(N)
    return(a,v,lbda,alpha,beta,gamma,t,z,s,w)

def make_af_Var(N,I,K):
    lbda = cp.Variable((N,K), nonneg = True)
    a = cp.Variable(I)
    v = cp.Variable(K, nonneg = True)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N)
    s = cp.Variable(N)
    w = cp.Variable(N)
    return(a,v,lbda,alpha,beta,gamma,t,s,w)

In [3]:
df_returns6 = pd.read_csv('6_Portfolios_2x3.csv', skiprows = 15)

In [4]:
df_returns = df_returns6[0:1144].copy()
df_returns['Date'] = pd.to_datetime(df_returns['Date'], format = '%Y%m')
for i in range(1, len(df_returns.columns)):
    df_returns[df_returns.columns[i]] = pd.to_numeric(df_returns[df_returns.columns[i]])
df_returns

,Date,SMALL LoBM,ME1 BM2,SMALL HiBM,BIG LoBM,ME2 BM2,BIG HiBM
0,1926-07-01,1.0874,0.9349,-0.0695,5.7168,1.9620,1.4222
1,1926-08-01,0.7030,1.2300,5.3842,2.7154,2.6930,6.3154
2,1926-09-01,-2.9117,-0.1303,-0.4374,1.4287,0.0704,-0.7967
3,1926-10-01,-3.8196,-4.5860,-2.0112,-3.5898,-2.3398,-4.0970
4,1926-11-01,3.1806,3.7233,2.0944,3.1292,2.8952,3.4614
...,...,...,...,...,...,...,...
1139,2021-06-01,5.6058,0.4400,-1.0979,4.8188,-1.2594,-4.0036
1140,2021-07-01,-5.5593,-1.8623,-3.6521,3.1048,-0.0099,-2.3000
1141,2021-08-01,2.3903,1.5124,2.6680,3.5667,1.4122,3.0371
1142,2021-09-01,-4.3421,-3.4661,0.6445,-5.4525,-3.8570,-0.2526


In [5]:
startdate = dt(1984,1,3)
enddate = dt(2014,1,3)
X = df_returns[np.logical_and(df_returns.Date >= startdate, df_returns.Date <= enddate )][df_returns.columns[1:7]]
X = X.reset_index(drop = True)
R = X.to_numpy()
R = R/100
r_f = 0.0007
W0 = 100

In [6]:
len(R)

360

In [7]:
def RC_powerutility_pmin(sets,p,R,r,phi_conj, h_conj,W0,par=1,par_u=1):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    c = cp.Variable(1)
    [a,v,lbda,alpha,beta,gamma,t,z,s] = make_RC_Var(N,M,I)
    constraints.append(cp.abs(a)<=W0)
    f_obj = 0
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        f_obj = (cp.power(W0*(1+(R @ a)[i]),rav)-1)/rav*p[i] + f_obj
        constraints.append((-cp.power(W0*(1+(R @ a)[i]),rav)-1)/rav - lbdasum - beta <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints = phi_conj(gamma,s[i],t[i],w[i],constraints)
    constraints = h_conj(lbda,v,z,par,constraints)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Minimize(c)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

In [8]:
def RC_exputility_pmin(sets,p,R,r,phi_conj, h_conj,W0,par=1,par_u=1):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    c = cp.Variable(1)
    [a,v,lbda,alpha,beta,gamma,t,z,s,w] = make_RC_Var(N,M,I)
    constraints=[cp.abs(a)<=W0, cp.sum(a)==1]
    #f_obj = 0
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        arg = -W0*(1+(R @ a)[i])/par_u
        #f_obj = (1-cp.exp(arg))*p[i] + f_obj
        constraints.append(-(1-cp.exp(arg)) - lbdasum - beta <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints = phi_conj(gamma,s[i],t[i],w[i],constraints)
    constraints = h_conj(lbda,v,z,par,constraints)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Minimize(c)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

def af_RC_exp_pmin(p,R,r,phi_conj,slope,const,W0,par = 2, par_u = 1):
    N = len(p)
    I = len(R[0])
    K = len(slope)
    c = cp.Variable(1)
    [a,v,lbda,alpha,beta,gamma,t,s,w] = make_af_Var(N,I,K)
    constraints = [cp.abs(a)<= W0, cp.sum(a) == 1]
    for i in range(N):
        #arg = -(R @ a)[i]/par_u
        arg = -W0*(1+(R @ a)[i])/par_u
        #constraints.append(-(R @ a)[i] - cp.sum(lbda[i]) - beta <= 0)
        constraints.append(-(1-cp.exp(arg)) - cp.sum(lbda[i]) - beta <= 0)
        constraints.append(s[i] == -alpha + lbda[i]@slope)
        constraints.append(lbda[i] <= v)
        constraints = phi_conj(gamma,s[i],t[i],w[i],constraints)
    constraints.append(alpha + beta + gamma * r  + v@const + p@t <= c)
    obj = cp.Minimize(c)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)


In [9]:
def RC_exputility_pmax(sets,p,R,r,c,r_f,phi_conj, h_conj,W0,par=2,par_u=1):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    [a,v,lbda,alpha,beta,gamma,t,z,s,w] = make_RC_Var(N,M,I)
    constraints=[cp.abs(a)<=W0]
    f_obj = 0
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        arg = -(W0*(1+(R @ a)[i]+(1-cp.sum(a))*r_f))/par_u
        f_obj = (1-cp.exp(arg))*p[i] + f_obj
        constraints.append(-(1-cp.exp(arg)) - lbdasum - beta <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints = phi_conj(gamma,s[i],t[i],w[i],constraints)
    constraints = h_conj(lbda,v,z,par,constraints)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

def af_RC_exp_pmax(p,R,r,r_f,c,phi_conj,slope,const,W0,par = 2, par_u = 1):
    N = len(p)
    I = len(R[0])
    K = len(slope)
    [a,v,lbda,alpha,beta,gamma,t,s,w] = make_af_Var(N,I,K)
    constraints = [cp.abs(a)<= W0]
    f_obj = 0
    for i in range(N):
        arg = -(W0*(1+(R @ a)[i]+(1-cp.sum(a))*r_f))/par_u
        f_obj = (1-cp.exp(arg))*p[i] + f_obj
        constraints.append(-(1-cp.exp(arg)) - cp.sum(lbda[i]) - beta <= 0)
        constraints.append(s[i] == -alpha + lbda[i]@slope)
        constraints.append(lbda[i] <= v)
        constraints = phi_conj(gamma,s[i],t[i],w[i],constraints)
    constraints.append(alpha + beta + gamma * r  + v@const + p@t <= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

In [10]:
import importlib
importlib.reload(ct)
importlib.reload(ut)
importlib.reload(af)
importlib.reload(hr)
importlib.reload(hf)

<module 'distortion_function' from 'C:\\Users\\gjin\\Robust\\distortion_function.py'>

In [11]:
e_tol = 0.001
#phi_func = phi.kb_cut
phi_func = phi.mod_chi2_cut
#phi_conj = phi.kb_conj
phi_conj = phi.mod_chi2_conj
#h_func = hf.h_lin_cut
h_func = hf.h_spw_cut
#h_eva = hf.h_lin_eva
h_eva = hf.h_spw_eva
#h_conj = hf.h_lin_conj
h_conj = hf.h_spw_conj
utility = ut.exp_utility
utility_eva = ut.exp_utility_eva
N=R.shape[0]
p = np.zeros(N)+1/N
I = R.shape[1]
#phi_dot=1
phi_dot = 2
r = phi_dot/(2*N)*scipy.stats.chi2.ppf(0.95, N-1)

In [13]:
r

1.1227281055463512

In [16]:
prob = hr.hit_and_run(p,hr.mod_chi2_calc,r,1,100)

1.0

In [49]:
cut_res = ct.cut_rob_pmin(R,p,e_tol,utility,utility_eva,h_func,phi_func,h_eva,r,W0,par = 2, par_u=1)
print(cut_res)

-0.6048781574145934 [-0.63788692] 0
-0.5834709527579616 [-0.62727413] 1
-0.6058665432160661 [-0.62289779] 2
-0.6101915643315174 [-0.61847693] 3
-0.6110425429929329 [-0.61598388] 4
-0.6070735867809587 [-0.61563469] 5
-0.6119050596305633 [-0.6149433] 6
-0.6124889339543217 [-0.61387883] 7
-0.6122610341680186 [-0.6131024] 8
(array([-1.        ,  1.        ,  0.32310474,  1.        ,  0.03570105,
       -0.35880579]), -0.6131023983478621, 8)


In [37]:
w = cut_res[0]
rank = np.argsort(R.dot(w))
sets = ct.ranktoset(rank)

In [23]:
af_RC_exp_pmin(p,R,r,phi_conj,slope,const+0.001,par = 2, par_u = 1)

NameError: name 'slope' is not defined

In [12]:
x_points = af.affine_approx_hspw(2,0.001)
[slope, const] = af.makepoints(af.sing_pw,x_points,par = 20)

In [15]:
W0 = 5

In [18]:
t1_af_res = process_time()
#af_res = af_RC_exp_pmin(p,R,r,phi_conj,np.array([1/(1-0.9),0]),np.array([0,1]),W0,par = 0.9, par_u = 100)
af_res = af_RC_exp_pmin(p,R,r,phi_conj,slope,const+0.001,W0,par = 20, par_u = 10)
t2_af_res = process_time()
print(af_res)
print('time', t2_af_res- t1_af_res)

(array([-0.91629958,  3.95109823, -3.28485973, -0.00966055,  0.78212771,
        0.47759392]), -0.3517096352759964)
time 4.859375


In [17]:
#af.res1 = af_RC_exp_pmin(p,R,r,phi_conj,slope,const+0.0001,par = 2, par_u = 1)
#print(af.res1)
w_af = af_res[0]
rank = np.argsort(W0*(1+R.dot(w_af)))
sets = ct.ranktoset(rank)
t1_rc_res = process_time()
rc_res = RC_exputility_pmin(sets,p,R,r,phi_conj, h_conj,W0,par=2,par_u=10)
t2_rc_res = process_time()
print(rc_res)
print('time', t2_rc_res - t1_rc_res)

(array([-1.55763529,  2.31217721, -0.3951753 ,  1.23521153, -0.18086841,
       -0.41370974]), -0.3786601778588288)
time 32.3125


In [17]:
res_nomaf = af_RC_exp_pmin(p,R,0,phi_conj,slope,const,W0,par = 20, par_u = 10)
print(res_nomaf)
w_nomaf = res_nomaf[0]
#x_nomaf = ut.lin_utility_eva(R,w_nomaf,W0,1)
x_nomaf = ut.exp_utility_eva(R,w_nomaf,W0,10)
[wc,q_b] = ct.robustcheck(x_nomaf,p,h_func,phi_func,r,20)  
print(wc)
print(np.max(q_b),np.min(q_b),np.mean(q_b))

(array([-1.51214523,  2.31824522, -0.45111411,  1.24112365, -0.15219193,
       -0.44391759]), -0.3747752117387939)
-0.3419785333621515
0.6867181995639305 5.746295832276204e-10 0.0027777777777777783


In [123]:
print(res_nomaf)

(array([-28.41016114,  37.24498428,   3.97635752,  15.63343188,
       -20.74004966,  -6.70456287]), -0.418504007718397)


In [19]:
tb = process_time()
res_nom = ct.cut_nom_pmin(R,p,0,utility,utility_eva,h_eva,W0,par=2,par_u=10)
te = process_time()
w_nom = res_nom[0]
x_nom = ut.exp_utility_eva(R,w_nom,W0,10)
print(ct.robustcheck(x_nom,p,h_func,phi_func,r,2))
print('time', te-tb)

-0.38064153438542675 [-0.40490619] 0
-0.38295583126845956 [-0.40108007] 1
-0.3833555076736778 [-0.39726323] 2
-0.3861236739393751 [-0.39616325] 3
-0.3912997580878683 [-0.39564891] 4
-0.3896320239692591 [-0.3950065] 5
-0.38929305120303564 [-0.39471632] 6
-0.3902974331380495 [-0.39431841] 7
-0.39034115659174534 [-0.39345943] 8
-0.39191307785886664 [-0.39338013] 9
-0.39020809072358054 [-0.39306676] 10
-0.3916842982257834 [-0.39280618] 11
-0.3913288989239551 [-0.39274334] 12
-0.39169697680166715 [-0.39258826] 13
-0.39135979224262896 [-0.39253511] 14
-0.3920802826411648 [-0.39251188] 15
-0.3911770130239924 [-0.39236533] 16
-0.39188049353097104 [-0.39234742] 17
-0.3920063357414098 [-0.3922708] 18
-0.3921041833652582 [-0.39226452] 19
-0.3918840828897145 [-0.39221068] 20
-0.3920832928363959 [-0.39220325] 21
-0.3920410167126207 [-0.39219494] 22
-0.3921192772689288 [-0.39219205] 23
-0.3921057778752627 [-0.39218346] 24
-0.39213223103454353 [-0.39218127] 25
-0.39208710808051905 [-0.39216306] 26
-0

C:\Users\gjin\Anaconda3\lib\site-packages\cvxpy\problems\problem.py:1278: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


-0.39214098932484903 [-0.39214294] 39
-0.3921395419587889 [-0.39214248] 40
-0.3921402823073906 [-0.39214237] 41
-0.3921410584402989 [-0.39214228] 42
-0.39214110545840514 [-0.39214219] 43
-0.39214134261557326 [-0.39214208] 44
-0.3921414269887382 [-0.39214204] 45
-0.3921412155647932 [-0.39214179] 46
-0.39214151390636215 [-0.39214175] 47
-0.39214143611694696 [-0.39214172] 48
-0.39214148312845465 [-0.39214169] 49
-0.39214152272789643 [-0.39214153] 50
-0.3921415588303241 [-0.39214151] 51
(-0.3779952180787354, array([7.01070093e-03, 3.30466658e-04, 8.50560363e-04, 5.54396027e-03,
       1.13801093e-03, 5.15681773e-04, 9.44531507e-05, 2.50196114e-04,
       8.80742969e-04, 1.94491000e-04, 2.12574461e-03, 2.07196361e-04,
       6.63289478e-04, 2.17192612e-03, 5.67620580e-04, 1.65038836e-04,
       1.40656404e-04, 7.85876450e-04, 1.61999594e-03, 1.62198737e-03,
       3.64442539e-04, 1.00656239e-05, 1.10372193e-04, 3.91507598e-03,
       3.44265073e-05, 2.37527338e-05, 5.33331126e-04, 2.2596790

In [20]:
x_nom = ut.exp_utility_eva(R,w_nom,W0,10)

In [21]:
[rbvalue_nom,qb_nom]=ct.robustcheck(x_nom,p,h_func,phi_func,r,2)
print(w_nom)
print(rbvalue_nom)
print(ct.nominal_risk(x_nom,p,h_eva,2)[0])

[-1.90479174  2.54809789  0.07950575  1.43778407 -0.62179049 -0.53880548]
-0.3779952180787354
-0.3921415588303241


In [57]:
w_nom

array([-1.90410303,  2.54628979,  0.08348333,  1.4399583 , -0.62301816,
       -0.54261023])

In [22]:
w_rob = rc_res[0]
x_rob = ut.exp_utility_eva(R,w_rob,W0,10)
[risk_robnom, qb_nomrb]=ct.nominal_risk(x_rob,p,h_eva,2)
print(risk_robnom)

-0.3918807225379179


In [49]:
w_nom

array([-1.90410303,  2.54628979,  0.08348333,  1.4399583 , -0.62301816,
       -0.54261023])

In [105]:
utility = ut.exp_utility_pmax
utility_eva = ut.exp_utility_eva_pmax
c = 0.02

In [86]:
t1_cutstart = process_time()
cut_res = ct.cut_rob_pmax(R,p,e_tol,utility,utility_eva,h_func,phi_func,h_eva,r, r_f, c, W0, par=2, par_u=200)
t1_cutstop = process_time()
print("cut-time:", t1_cutstop-t1_cutstart) 
print(cut_res)
w_cut = cut_res[0]
rank = np.argsort(W0*(1+R.dot(w_cut)+(1-np.sum(w_cut))*r_f))
sets = ct.ranktoset(rank)
t1_resct = process_time()
rc_resmax = RC_exputility_pmax(sets,p,R,r,c,r_f,phi_conj, h_conj,W0,par=2,par_u=200)
t2_resct = process_time()
print('res-time',t2_resct-t1_resct)
print(rc_resmax)
x_rob = ut.exp_utility_eva_pmax(R,r_f,rc_resmax[0],W0,200)
print(ct.robustcheck(x_rob,p,h_func,phi_func,r,2)[0])

0.07754350224598186 0.02 0
0.020472059982278613 0.02 1
cut-time: 16.984375
(array([-41.47371081,  55.83687808,   0.41341288,  26.59427596,
       -16.12152565, -10.65382054]), 0.4477138709398567, 1)
res-time 32.9375
(array([-41.41842187,  55.7936817 ,   0.39746181,  26.56581941,
       -16.10659185, -10.64642667]), 0.44770904082638724)
0.019995732887310727


In [70]:
x_rob = ut.exp_utility_eva_pmax(R,r_f,rc_resmax[0],W0,200)
print(ct.robustcheck(x_rob,p,h_func,phi_func,r,2)[0])

-0.3000016071202275


In [96]:
t1_af = process_time()
af_pmax_res = af_RC_exp_pmax(p,R,r,r_f,c,phi_conj,slope,const,W0,par = 2, par_u = 200)
t2_af = process_time()
print(af_pmax_res)
print('seconds', t2_af-t1_af)
t1_af_l = process_time()
af_pmax_res_l = af_RC_exp_pmax(p,R,r,r_f,c,phi_conj,slope,const+0.001,W0,par = 2, par_u = 200)
t2_af_l = process_time()
print(af_pmax_res_l)
print('seconds low', t2_af_l-t1_af_l)
w_af_rob = af_pmax_res[0]
rank = np.argsort(W0*(1+R.dot(w_af_rob)+(1-np.sum(w_af_rob))*r_f))
sets = ct.ranktoset(rank)
t1_resct = process_time()
rc_robmax = RC_exputility_pmax(sets,p,R,r,c,r_f,phi_conj, h_conj,W0,par=2,par_u=200)
t2_resct = process_time()
print(rc_robmax)
print('res-time',t2_resct-t1_resct)

(array([-41.45716   ,  55.81420784,   0.44551949,  26.61893252,
       -16.10308739, -10.69012328]), 0.44772711598820836)
seconds 5.96875
(array([-41.34189433,  55.65820961,   0.43623302,  26.52841499,
       -16.06510988, -10.64736032]), 0.44770022010098304)
seconds low 6.15625
(array([-41.36258815,  55.69373123,   0.4380042 ,  26.54956699,
       -16.07116502, -10.6581975 ]), 0.4477091583929269)
res-time 34.0625


In [113]:
c = -0.35

In [114]:
nom_afres = af_RC_exp_pmax(p,R,0,r_f,c,phi_conj,slope,const,W0,par = 2, par_u = 200)
print(nom_afres)
w_af_nom = nom_afres[0]
nom_afres1 = af_RC_exp_pmax(p,R,0,r_f,c,phi_conj,slope,const+0.001,W0,par = 2, par_u = 200)
print(nom_afres1)
#rank = np.argsort(W0*(1+R.dot(w_af_nom)+(1-np.sum(w_af_nom))*r_f))
#sets = ct.ranktoset(rank)
#rc_nommax = RC_exputility_pmax(sets,p,R,0,c,r_f,phi_conj, h_conj,W0,par=2,par_u=200)
#print(rc_nommax)
x_nommax = ut.exp_utility_eva_pmax(R,r_f,nom_afres1[0],W0,200)
print(ct.robustcheck(x_nommax,p,h_func,phi_func,r,2)[0])

(array([-32.3426664 ,  43.42543771,   0.77405173,  21.48525953,
       -12.25559462,  -8.91177141]), 0.44462053481242947)
(array([-31.56516227,  42.35723669,   0.72068318,  20.93534621,
       -11.99941509,  -8.6270416 ]), 0.4440622985745267)
-0.10057619437608273


In [56]:
cut_resnom = ct.cut_nom_pmax(R,r_f,c,p,0,utility,utility_eva,h_eva,par=2,par_u=1)
print(cut_resnom)

0.06552169495527872 0.01 0
0.024843918806091728 0.01 1
0.01831306988775344 0.01 2
0.013061839135007887 0.01 3
0.011846403265971504 0.01 4
0.0104940704009597 0.01 5
0.010181607098847018 0.01 6
0.010127859141128362 0.01 7
0.010054387922941568 0.01 8
0.010048966571139234 0.01 9
0.010017238430731509 0.01 10
0.010008596938737681 0.01 11
0.010004288322538175 0.01 12
0.010002256425781385 0.01 13


C:\Users\gjin\Anaconda3\lib\site-packages\cvxpy\problems\problem.py:1278: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


0.010002238026797847 0.01 14
0.01000096110094924 0.01 15
0.01000090318977667 0.01 16
0.010000509534410285 0.01 17
0.010000267140860294 0.01 18
0.010000291320855762 0.01 19
0.010000115384995368 0.01 20
0.010000076143165842 0.01 21
0.010000030790091832 0.01 22
0.009999995624743725 0.01 23
(array([-3.92002824,  5.21943564,  0.20624784,  2.77385403, -1.53742535,
       -1.09787656]), 0.03202892219540232, 23)
